In [1]:
import torch
from torch.utils.data import DataLoader, Dataset

In [2]:
class HiCDataset(Dataset):
    def __init__(self, data_files):
        self.data = []  # To store all sequences and HiC vectors
        
        # Load and process the data files
        for file in data_files:
            file_data = torch.load(file, weights_only=True)
            
            for data in file_data:
                ohe_sequence, hic_vector = data

                # Process the OHE sequence
                ohe_sequence = ohe_sequence.squeeze(0)  # Remove singleton dimension
                
                # Ensure the sequence has the correct shape
                # assert ohe_sequence.shape[0] == 4, f"Expected 4 channels, but got {ohe_sequence.shape[0]}"
                # assert len(ohe_sequence.shape) == 2, f"Expected 2D shape for sequence, but got {ohe_sequence.shape}"
                
                # Add processed pair to the data list
                self.data.append((ohe_sequence, hic_vector))
        
        print(f"Total sequences loaded: {len(self.data)}")
        
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Fetch the preprocessed (ohe_sequence, hic_vector) pair from memory
        ohe_sequence, hic_vector = self.data[idx]
        return ohe_sequence, hic_vector

In [3]:
train_dataset = HiCDataset(['/scratch1/smaruj/background_generation/training_pt_files/background_fold0_0.pt'])

Total sequences loaded: 100


In [4]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=4, pin_memory=True)

In [5]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

# from model import SeqNN
from model_v2_compatible import SeqNN


In [6]:
device = torch.device("cuda")

In [7]:
model = SeqNN().to(device)

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
# Helper function to set diagonal elements to a specific value
def set_diag(matrix, value, k):
    # Explicitly set the diagonal to 'value' (in this case, np.nan) for each k
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value


def from_upper_triu(vector_repr, matrix_len, num_diags):
    # Ensure vector_repr is a NumPy array (if it's a PyTorch tensor, convert it)
    if isinstance(vector_repr, torch.Tensor):
        vector_repr = vector_repr.detach().flatten().cpu().numpy()  # Flatten and convert to NumPy array

    # Initialize a zero matrix of shape (matrix_len, matrix_len)
    z = np.zeros((matrix_len, matrix_len))

    # Get the indices for the upper triangular matrix
    triu_tup = np.triu_indices(matrix_len, num_diags)

    # Assign the values from the vector_repr to the upper triangular part of the matrix
    z[triu_tup] = vector_repr

    # Set the diagonals specified by num_diags to np.nan
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)

    # Ensure the matrix is symmetric
    return z + z.T

In [10]:
def plot_map(matrix, vmin=-0.6, vmax=0.6, palette="RdBu_r", width=5, height=5):
    fig, axes = plt.subplots(1, 1, figsize=(width, height))

    sns.heatmap(
        matrix,
        vmin=vmin,
        vmax=vmax,
        cbar=False,
        cmap=palette,
        square=True,
        xticklabels=False,
        yticklabels=False,
        ax=axes
    )

    plt.tight_layout()
    plt.show()

In [11]:
import torch.nn.functional as F

In [12]:
model.train() # doesn't work since batch norm requires two examples
# model.eval()

train_loss = 0  # Initialize training loss
for batch_idx, (data, target) in enumerate(train_loader):
    data, target = data.to(device), target.to(device)
    output = model(data)
    
    print("data shape", data.shape)
    print("target shape", target.shape)
    print("output shape", output.shape)
    
    if torch.any(torch.isnan(target)):
        print("NaNs in target")
        
        matrix_np = from_upper_triu(target.cpu().numpy(), matrix_len=448, num_diags=2)
        plot_map(matrix_np)
        
    if torch.any(torch.isnan(output)):  
        print("NaNs in model output")
    
    loss = F.mse_loss(output, target)
    print(loss)

data shape torch.Size([2, 4, 1310720])
target shape torch.Size([2, 1, 130305])
output shape torch.Size([2, 1, 130305])
tensor(0.7773, device='cuda:0', grad_fn=<MseLossBackward0>)
data shape torch.Size([2, 4, 1310720])
target shape torch.Size([2, 1, 130305])
output shape torch.Size([2, 1, 130305])
tensor(0.7627, device='cuda:0', grad_fn=<MseLossBackward0>)
data shape torch.Size([2, 4, 1310720])
target shape torch.Size([2, 1, 130305])
output shape torch.Size([2, 1, 130305])
tensor(0.7108, device='cuda:0', grad_fn=<MseLossBackward0>)
data shape torch.Size([2, 4, 1310720])
target shape torch.Size([2, 1, 130305])
output shape torch.Size([2, 1, 130305])
tensor(0.6852, device='cuda:0', grad_fn=<MseLossBackward0>)
data shape torch.Size([2, 4, 1310720])
target shape torch.Size([2, 1, 130305])
output shape torch.Size([2, 1, 130305])
tensor(1.0353, device='cuda:0', grad_fn=<MseLossBackward0>)
data shape torch.Size([2, 4, 1310720])
target shape torch.Size([2, 1, 130305])
output shape torch.Size([2